# **LiminalGPT Stage 2**: _Self attention_

A basic transformer based neural network trained on a literary text corpus.

In Stage 2, we explore the concept of self attention in a transformer architecture.

LiminalGPT is based on [Vaswani et al. (2017)](https://arxiv.org/pdf/1706.03762).


In [1]:
import torch
from typing import Final

SEED: Final[int] = 3433

torch.manual_seed(SEED);

## Building the intuition behind self attention

We will compute a running average of feature vectors for each token position. For a token at position `i`, we average the features from all tokens at positions `0` through `i` (inclusive). This creates a representation that captures the "context" or "history" leading up to each token.

Given a tensor of shape `(B, T, C)` where:

- `B` = batch size
- `T` = sequence length (time steps)
- `C` = number of channels (feature dimensions)

For each token, we want to aggregate information only from previous tokens and itself (not future tokens).


### Version 1: weighted aggregation via manual loops


In [2]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

For now, let us perform the weighted aggregation for each token via loops:


In [3]:
xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        # extract running history of tokens
        xprev = x[b, : t + 1]  # shape (T, C)
        # calculate running mean
        xbow[b, t] = xprev.mean(0)

In [4]:
x[0]

tensor([[-1.4970, -0.6357],
        [-0.6095, -0.1586],
        [ 0.9341, -0.0361],
        [-0.4688, -0.0288],
        [-1.2407,  0.2724],
        [ 0.1070, -1.9325],
        [ 0.8531, -0.5161],
        [ 0.0290, -0.1809]])

In [5]:
xbow[0]

tensor([[-1.4970, -0.6357],
        [-1.0532, -0.3972],
        [-0.3908, -0.2768],
        [-0.4103, -0.2148],
        [-0.5764, -0.1174],
        [-0.4625, -0.4199],
        [-0.2745, -0.4336],
        [-0.2366, -0.4020]])

### Version 2: weighted aggregation via matrix multiplication


#### 2.a Calculating global mean


In [6]:
a = torch.ones(3, 3)
print(f"{a = }")

b = torch.randint(0, 10, (3, 2)).float()
print(f"\n{b = }")

c = a @ b
print(f"\n{c = }")

mean = c.mean(0)
print(f"\n{mean = }")

a = tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])

b = tensor([[1., 4.],
        [4., 9.],
        [0., 4.]])

c = tensor([[ 5., 17.],
        [ 5., 17.],
        [ 5., 17.]])

mean = tensor([ 5., 17.])


#### 2.b Calculating running mean via `torch.tril()`


[torch.tril()](https://docs.pytorch.org/docs/stable/generated/torch.tril.html) converts all elements above the diagonal to 0:


In [7]:
ex = torch.ones(3, 3)
print(f"before tril():\n{ex}")
print("\n")
ex = torch.tril(ex)
print(f"after tril():\n{ex}")

before tril():
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])


after tril():
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])


We can use `tril()` in matrix multiplication to calculate the running mean:


In [8]:
a = torch.tril(torch.ones(3, 3))
print(f"{a = }")

b = torch.randint(0, 10, (3, 2)).float()
print(f"\n{b = }")

c = a @ b
print(f"\n{c = }")

a = tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

b = tensor([[9., 1.],
        [5., 9.],
        [8., 4.]])

c = tensor([[ 9.,  1.],
        [14., 10.],
        [22., 14.]])


But how do we calculate the mean from the running sums?

We can perform a mean by **normalizing the rows** of tensor `a`:


In [9]:
a = a / a.sum(1, True)
a

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

We know:

$$mean = \frac{1}{\text{total\ elements}}\times\text{sum of elements} $$

For a running average, we want each row of `a` to be weights that sum to 1, so the result is the average and not the sum.

In the example above:

Row 0: [1.0, 0, 0] → sums to 1 → averages token 0 only

Row 1: [0.5, 0.5, 0] → sums to 1 → averages tokens 0-1

Row 2: [0.33, 0.33, 0.33] → sums to 1 → averages tokens 0-2

Now when we do `a @ b`, we get the running mean instead of the running sum:

- Position 0:
  $$1.0 × b[0] = \textcolor{lightblue}{\frac{1}{1} \times (b[0])} =  \text{just b[0]}$$

- Position 1:
  $$0.5 × b[0] + 0.5 × b[1] = \textcolor{lightblue}{\frac{1}{2} \times (b[0] + b[1])} = \text{mean of } b[0] \text{ and } b[1]$$

- Position 2:
  $$0.33 × b[0] + 0.33 × b[1] + 0.33 × b[2] = \textcolor{lightblue}{\frac{1}{3} \times (b[0] + b[1] + b[2])} = \text{mean of all three}$$

Observe how these $\textcolor{lightblue}{\text{expressions}}$ match our _mean_ formula!

Therefore, if we normalize the weights so that they sum to 1, we transform the matrix multiplication from computing a weighted sum into computing a **weighted average**. This is exactly what our manual loop implementation was trying to do!


#### 2.c Complete vectorized method


In [10]:
# version 1
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        # extract running history of tokens
        xprev = x[b, : t + 1]  # shape (T, C)
        # calculate running mean
        xbow[b, t] = xprev.mean(0)

In [11]:
# version 2
weights = torch.tril(torch.ones(T, T))  # reason for shape explained below
weights = weights / weights.sum(1, True)
weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [12]:
#  (T, T) @ (B,T,C) --> (B, T, T) @ (B, T, C) --> (B,T,C)
# we want the output to have same shape as x so weights shape must be (T,T)
xbow2 = weights @ x
print("xbow and xbow2 have close values:", torch.allclose(xbow, xbow2))
# compare batch 1
print("\nfirst batch of xbow and xbow2:")
xbow[0], xbow2[0]

xbow and xbow2 have close values: True

first batch of xbow and xbow2:


(tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]),
 tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]))

### Version 3: weighted aggregation via softmax


Read about [torch.Tensor.masked_fill()](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill_.html).

This version achieves the same aggregation effect but helps us reason about the mechanism in a different pov.

---

In all examples, we intitialized weight matrix to zero to demonstrate the mechanics of aggregation.

In practice, these weights are data-dependent and learned from the input.
We can think of their magnitude as interaction strength or affinity between tokens:

- Higher weights mean stronger connections between token pairs
- Lower weights mean weaker connections

Let sequence: ["The", "cat", "sat"]

For this sequence we would have a 3x3 weight matrix with one entry for every possible pair of tokens in the sequence:

- w[0,0]: pair ("The" → "The") - how much "The" attends to itself
- w[1,0]: pair ("cat" → "The") - how much "cat" attends to "The"
- w[1,2]: pair ("cat" → "sat") - how much "cat" attends to "sat" (this would be masked to -inf in causal attention since "sat" comes after "cat")

The weights determine how much each token _pays attention_ to other tokens. :)

---

The masked fill operation can be thought of as a clamping operation that ensures tokens from the future cannot communicate/aggregate.

By setting future positions to -inf before softmax, we enforce causality:

- Token at position t can only attend to tokens at positions 0 through t (including itself)
- Token at position t cannot "see" tokens at positions t+1, t+2, ..., T-1

We must prevent information leakage from future tokens. Otherwise, the model can "cheat" as it has information about what comes next.

The mask ensures that the model learns to predict each token using only the context that came before it.


In [13]:
tril = torch.tril(torch.ones(T, T))

weights = torch.zeros(T, T)
weights = weights.masked_fill(tril == 0, float("-inf"))
weights

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

Softmax-ing along dim 1 gives us the same weight tensor as in version 2:

Softmax exponentiates each element and divides each element by the sum of all elements. Here:

- $\exp(-\infty) = 0$
- $\exp(0) = 1$


In [14]:
weights = weights.softmax(1)
weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [15]:
# same aggregation as v2 via matrix mul
xbow3 = weights @ x

print("xbow and xbow3 have close values:", torch.allclose(xbow, xbow3))
# compare batch 1
print("\nfirst batch of xbow and xbow3:")
xbow[0], xbow3[0]

xbow and xbow3 have close values: True

first batch of xbow and xbow3:


(tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]),
 tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]))

## Building self attention


### Keys, Values and Queries

In self-attention, keys `k`, queries `q`, and values `v` (all shaped B, T, head_size) are three different vector representations of a token. Keys and queries work together to compute the attention weights, which encapsulate token affinity. Values contain the actual information that gets aggregated.

The dimension of these vectors is determined by hyperparameter `head_size`:

$$head\_size = \frac{\text{n\_embd}}{\text{num\_heads}}$$

- **Query**: A vector that represents what information a token wants to find from other tokens in a sequence.

- **Key**: A vector that represents what information a token has to offer.

- **Value**: A vector that represents the actual information content that will be passed forward when this token is attended to.


Remember that our input tokens start with embeddings in `C`-dimensional space. These embeddings capture general information about each token.

But for the concept of "attention", we need a mechanism that quantifies the aggregated information (via a score) about _contextually-similar_ tokens. Our current embeddings do not possess that capability by itself.

Let us transform our token embeddings into three separate **learnable** "semantic spaces" where such comparisons can happen:

- Weight $W_{key}$ projects the `C`-embeddings into a _key space_ while emphasizing the characteristics that make this token useful to others and learning to extract the features that it offers to other tokens.

- Weight $W_{query}$ projects the `C`-embeddings into a _query space_ while emphasizing the requirements of this token and learning to extract the features that it is searching for.

- Weight $W_{value}$ projects the `C`-embeddings into a _value space_ while learning to extract and transform the actual information that will be communicated and aggregated. The value is what gets passed along when attention is paid to this token.

If we used the same transformation for both keys and queries (or no transformation at all), every token would be asking for exactly what it offers, which limits the model's ability to learn complex relationships.

So, we have three separate learned matrices for each token to ensure that:

- A token can need something different from what it provides
- The model learns asymmetric relationships
- What gets communicated (value) can be different from both what's offered (key) and what's requested (query)

Consider this example:

token = "cat" and embedding = e_cat

```py
k_cat = e_cat @ W_key    # "I represent an animal/subject"
q_cat = e_cat @ W_query  # "I'm looking for actions that involve me"
```

token = "sat" and embedding = e_sat

```py
k_sat = e_sat @ W_key    # "I represent a past-tense action"
q_sat = e_sat @ W_query  # "I'm looking for the subject doing this action"
```

When we compute `q_sat · k_cat` (query of "sat" with key of "cat"), we get a high score because:

- "sat" is asking for a subject

- "cat" is offering subject information

The weight matrices learn these transformations during training to maximize the model's ability to predict the next token correctly.


In [17]:
import torch.nn as nn

B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# single Head performing self-attention

# number of dims for key and query vector
head_size = 16

# project token embeddings into key and query spaces
key = nn.Linear(C, head_size, bias=False)  # x @ W_key
query = nn.Linear(C, head_size, bias=False)  # x @ W_query
value = nn.Linear(C, head_size, bias=False)  # x @ W_value

k, q, v = key(x), query(x), value(x)  # all (B, T, 16)

# build the attention-weight 2d matrix
# (B, T, 16) @ (B, 16, T) --> (B, T, T)
weights = q @ k.transpose(-2, -1)
# Note: in practice, we scale by 1/sqrt(head_size) to prevent large dot products:
# weights = weights * (head_size ** -0.5)

# mask for causal attention (decoder-style)
mask = torch.tril(torch.ones(T, T))
# make upper triangle of weights all -inf (so tokens can't see future)
weights = weights.masked_fill(mask == 0, float("-inf"))
# row normalization:
# upper diagonal of weights also all zeros from softmax
# each row sums to 1

attention_matrix = weights.softmax(-1)
attention_matrix.shape, attention_matrix[0]

(torch.Size([4, 8, 8]),
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.8422, 0.1578, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3402, 0.4728, 0.1870, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4181, 0.0994, 0.0543, 0.4282, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2376, 0.3139, 0.1179, 0.1745, 0.1561, 0.0000, 0.0000, 0.0000],
         [0.1877, 0.0037, 0.0105, 0.7500, 0.0084, 0.0397, 0.0000, 0.0000],
         [0.0639, 0.1229, 0.1331, 0.1273, 0.0887, 0.4253, 0.0387, 0.0000],
         [0.0103, 0.2044, 0.3888, 0.0068, 0.2348, 0.1315, 0.0166, 0.0067]],
        grad_fn=<SelectBackward0>))

In [18]:
# matrix mul weight aggregation
# attention weights determine HOW MUCH of each token's value to aggregate
# (B,T,T) @ (B,T,head_size) --> (B,T,head_size)
weighted_mean = attention_matrix @ v
weighted_mean.shape, weighted_mean[0]

(torch.Size([4, 8, 16]),
 tensor([[-0.7873, -0.9653,  0.0882,  0.3355,  0.3470,  0.2018,  0.1338,  1.0050,
           0.3033,  0.1344,  0.1567, -0.3098,  0.5923, -0.3762,  0.6779,  1.1004],
         [-0.6789, -0.8748, -0.0338,  0.1613,  0.2440,  0.2199,  0.1627,  0.7748,
           0.2213,  0.2339,  0.1822, -0.0267,  0.4353, -0.3623,  0.7446,  0.8694],
         [-0.5147, -0.4916, -0.4845, -0.2215,  0.0669,  0.2078,  0.0515,  0.1852,
          -0.0291,  0.4868,  0.0566,  0.7110,  0.0208, -0.1700,  0.7195,  0.2381],
         [-0.7717, -0.4857, -0.1452,  0.1852,  0.2286, -0.3401,  0.1301,  0.5450,
          -0.0379, -0.0999,  0.2188,  0.2203,  0.3416, -0.2148,  1.0525,  0.3324],
         [-0.6667, -0.3456, -0.3884,  0.0131,  0.0771, -0.0797,  0.1321,  0.2470,
          -0.0718,  0.4088, -0.0803,  0.4958,  0.1205, -0.1099,  0.8937,  0.0953],
         [-0.7799, -0.2646, -0.0524,  0.2723,  0.2300, -0.7783,  0.1507,  0.4451,
          -0.1754, -0.3977,  0.3281,  0.2402,  0.3407, -0.1515,  1.2